# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible from:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and their `@id` values.

In [ ]:
# Identify record sets in the dataset

from pprint import pprint

record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, type: {f.data_type})")
    print()

## 3. Data Extraction

Load data from specific record set(s) into DataFrames for analysis.

Use the record set and field `@id`s from the overview above for precise extraction.

In [ ]:
# List the record set @ids to extract records from
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet '{record_set_id}' with shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    if len(df.columns) <= 8:
        display(df.head())
    else:
        display(df.iloc[:5, :8])  # Show first 8 columns only for preview
    print()

## 4. Exploratory Data Analysis (EDA)

Let's select one record set, choose a numeric field (by `@id`), perform filtering, normalization, and grouping.

In [ ]:
# Select a record set and a numeric field (by @id)
# Example using the first record set and its first numeric field

target_record_set = dataset.metadata.record_sets[0]
target_record_set_id = target_record_set.id

# Find the first numeric field in this record set
numeric_field = None
for field in target_record_set.fields:
    if field.data_type in ["schema:Number", "schema:Float", "schema:Integer"]:
        numeric_field = field.id
        break

if numeric_field is None:
    raise Exception("No numeric field found in the first record set.")

# Use the DataFrame loaded previously
df = dataframes[target_record_set_id]

print(f"Analyzing RecordSet '@id': {target_record_set_id}")
print(f"Using numeric field '@id': {numeric_field}")

# Convert to numeric in case the field was loaded as string
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filtering by value > threshold
threshold = df[numeric_field].mean()  # Example: use mean as threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field (standard score)
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (if any)
group_field = None
for field in target_record_set.fields:
    if field.data_type in ["schema:Text", "schema:String"]:
        group_field = field.id
        break

if group_field and group_field in df.columns:
    print(f"Grouping filtered records by '{group_field}' and showing mean of numeric columns:")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (Filtered)")
plt.xlabel(numeric_field)
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (Normalized, Filtered)")
plt.xlabel(f"{numeric_field}_normalized")
plt.show()

# If grouping field exists, visualize mean by group
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(12,6))
    group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field} (Filtered)")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading a FAIR² Croissant dataset, inspecting its record sets and fields, extracting and preparing data using `mlcroissant`, filtering and normalizing a quantitative field, and producing basic exploratory visualizations. 

This workflow provides a foundation for more advanced analyses and serves as an example for working with other Croissant-compliant datasets.